In [1]:
import requests
import json
import pandas as pd

In [2]:
API_key = "1fd00ba235bf47278e2c13643565a3ce"

Проверяем работу ключа, совершаем 1-й запрос и смотрим, что пришло на выходе:

In [3]:
#1-й запрос
import requests
url = ('https://newsapi.org/v2/everything?''q=Apple&''apiKey=1fd00ba235bf47278e2c13643565a3ce')
response = requests.get(url)
res = response.json()


Смотрим на ключи, содержащиеся в словаре:

In [4]:
res.keys()

dict_keys(['status', 'totalResults', 'articles'])

Ключ 'articles' с точки зрения информативности содержит больше пользы:

- `source` — наименованиеисточника
- `author` — автор статьи
- `title` — название статьи
- `description` — описание статьи
- `url` — ссылка на статью
- `urlToImage` — название статьи
- `publishedAt` — дата публикации статьи
- `content` — текст статьи


In [5]:
res['articles'][0].keys()

dict_keys(['source', 'author', 'title', 'description', 'url', 'urlToImage', 'publishedAt', 'content'])

Убираем колонку, которая не содержит полезную информацию:

In [6]:
news_df = pd.DataFrame(res['articles'])
news_df = news_df.drop('urlToImage', axis=1)
news_df

,source,author,title,description,url,publishedAt,content
0,"{'id': 'the-verge', 'name': 'The Verge'}",Verge Staff,Rank the 50 best Apple products,"In honor of Apple’s 50th anniversary, The Verg...",https://www.theverge.com/cs/tech/900477/apple-...,2026-03-27T10:00:00Z,"For any one person, ranking 50 items can be a ..."
1,"{'id': 'the-verge', 'name': 'The Verge'}",Jason Snell,Between Jobs,This is part of our package about Apple's 50th...,https://www.theverge.com/tech/897520/apple-wit...,2026-03-30T17:57:09Z,<ul><li></li><li></li></ul>\r\nWith Steve Jobs...
2,"{'id': 'the-verge', 'name': 'The Verge'}",Emma Roth,Apple could put ads in Maps as soon as this su...,Apple will soon bring advertisements to its Ma...,https://www.theverge.com/tech/899053/apple-map...,2026-03-23T18:54:44Z,<ul><li></li><li></li><li></li></ul>\r\nThe ro...
3,"{'id': 'wired', 'name': 'Wired'}",Noam Scheiber,How the Vision Pro Rollout Inflamed Tensions a...,"Even before the headset’s release, the workfor...",https://www.wired.com/story/book-excerpt-mutin...,2026-04-07T10:00:00Z,"To roll out its new mixed-reality headset, the..."
4,"{'id': 'the-verge', 'name': 'The Verge'}",Jennifer Pattison Tuohy,Nuki adds Apple Home Key to its smart lock,"This week, my top pick for a retrofit smart lo...",https://www.theverge.com/tech/899330/nuki-keyp...,2026-03-24T13:02:13Z,<ul><li></li><li></li><li></li></ul>\r\nThe Ke...
...,...,...,...,...,...,...,...
64,"{'id': 'wired', 'name': 'Wired'}",Nena Farrell,Ikea’s New Lineup of Smart Home Gear Is Quietl...,"Ikea’s latest light bulbs, remotes, and more h...",https://www.wired.com/story/ikea-matter-smart-...,2026-04-08T13:16:00Z,I've always been an Ikea fan. I lived in nine ...
65,"{'id': None, 'name': 'MacRumors'}",Juli Clover,Apple Adds 'Genius Browse' Movie and TV Recomm...,tvOS 26.4 includes a new Genius Browse section...,https://www.macrumors.com/2026/03/18/tvos-26-4...,2026-03-18T23:17:12Z,tvOS 26.4 includes a new Genius Browse section...
66,"{'id': None, 'name': 'Gizmodo.com'}",Kyle Barr,There’s a Good Reason the MacBook Neo Is Apple...,Apple's $600 MacBook Neo finally eschews glue ...,https://gizmodo.com/theres-a-good-reason-the-m...,2026-03-16T16:10:24Z,The MacBook Neo took the scene last week as Ap...
67,"{'id': None, 'name': 'MacRumors'}",Joe Rossignol,No Major Apple Watch Redesign Expected This Year,In addition to indicating that a new full-size...,https://www.macrumors.com/2026/03/26/no-major-...,2026-03-26T18:12:55Z,In addition to indicating that a new full-size...


In [7]:
!pip install vaderSentiment

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.0/126.0 kB 4.0 MB/s eta 0:00:00


In [8]:
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
df = news_df
analyzer = SentimentIntensityAnalyzer()


In [9]:
def get_vader_sentiment(text):
    if pd.isna(text) or text == "None":
        return None
    return analyzer.polarity_scores(str(text))['compound']

df['content_sentiment'] = df['content'].apply(get_vader_sentiment)

print(df[['title','content_sentiment']])

                                                title  content_sentiment
0                     Rank the 50 best Apple products             0.4215
1                                        Between Jobs            -0.1585
2   Apple could put ads in Maps as soon as this su...             0.5574
3   How the Vision Pro Rollout Inflamed Tensions a...             0.3230
4          Nuki adds Apple Home Key to its smart lock            -0.7065
..                                                ...                ...
64  Ikea’s New Lineup of Smart Home Gear Is Quietl...             0.1655
65  Apple Adds 'Genius Browse' Movie and TV Recomm...             0.0000
66  There’s a Good Reason the MacBook Neo Is Apple...             0.4927
67   No Major Apple Watch Redesign Expected This Year             0.0000
68  Apple Can Create Smaller On-Device AI Models F...             0.0000

[69 rows x 2 columns]


In [10]:
df['content_sentiment'].mean()

np.float64(0.2209869565217391)

Какой вывод можно сделать?

Сегодня сентимент для Apple 0.22, что является положительным, но не сильно. Варимция у нас от [-1 до 1]


In [11]:
#2-й запрос
import requests
url = ('https://newsapi.org/v2/top-headlines?''category=business&''apiKey=1fd00ba235bf47278e2c13643565a3ce')
response = requests.get(url)
res = response.json()
news_df = pd.DataFrame(res['articles'])
news_df = news_df.drop('urlToImage', axis=1)



Забираем из 'source' только 'name'. 'id' - убираем

In [12]:
def clean_page_columns(news_df):
    news_df['source'] = news_df['source'].apply(lambda d: d['name'])
    return news_df[['source', 'author', 'title', 'description', 'url', 'publishedAt', 'content']]
page = pd.DataFrame(res['articles'])
news_df_cleaned = clean_page_columns(page)
news_df_cleaned

,source,author,title,description,url,publishedAt,content
0,TechCrunch,Tim Fernholz,Is Anthropic limiting the release of Mythos to...,Anthropic said this week that it limited the r...,https://techcrunch.com/2026/04/09/is-anthropic...,2026-04-09T18:50:05Z,Anthropic said this week that it limited the r...
1,Business Insider,Samuel O'Brient,'Big Short' investor Michael Burry says Anthro...,Palantir's stock is down. Michael Burry said W...,https://www.businessinsider.com/michael-burry-...,2026-04-09T18:43:00Z,Michael Burry has long been bearish on Palanti...
2,The Wall Street Journal,Laura Cooper,Exclusive | Sazerac Eyes Deal With Jack Daniel...,Interest comes as Pernod Ricard has been discu...,https://www.wsj.com/business/sazerac-eyes-deal...,2026-04-09T18:05:00Z,None
3,The Information,"Theo Wayt, Cory Weinberg, Valida Pau",SpaceX Woos Investors As C-Suite Shakeups Cont...,Some of the biggest Wall Street fund managers ...,https://www.theinformation.com/articles/spacex...,2026-04-09T17:53:00Z,None
4,Investor's Business Daily,None,Amazon Rises Toward Key Level; CEO Says This I...,Andy Jassy touted growth for the tech giant's ...,https://www.investors.com/news/technology/amaz...,2026-04-09T17:52:00Z,Information in Investors Business Daily is for...
5,Variety,Todd Spangler,David Zaslav Paramount Deal 'Windfall' Payout:...,ISS recommends Warner Bros. Discovery sharehol...,https://variety.com/2026/film/news/david-zasla...,2026-04-09T17:47:00Z,Does David Zaslav deserve to receive more than...
6,Associated Press,Susan Haigh,US Postal Service to suspend employer payments...,The U.S. Postal Service has decided to tempora...,https://apnews.com/article/mail-usps-pensions-...,2026-04-09T17:13:00Z,The U.S. Postal Service said Thursday it has i...
7,Mashable,"Chance Townsend, Timothy Beck Werth","Mark Zuckerberg announces Muse Spark, a new Me...",Nine months after founding Meta Superintellige...,https://mashable.com/article/mark-zuckerberg-m...,2026-04-09T17:06:34Z,None
8,SFGate,Ariana Bindman,'No idea what he’s thinking': New policy at Ph...,"Philz, the coffee chain that was founded in Sa...",https://www.sfgate.com/food/article/philz-outr...,2026-04-09T16:25:29Z,Customers enter Philz Coffee at Bay Street Eme...
9,Associated Press,"Rio Yamat, Ap Airlines, Travel Writer",Travelers face higher costs and fewer flight o...,Air travelers are facing a new reality of high...,https://apnews.com/article/airline-tickets-fee...,2026-04-09T16:22:00Z,A new reality is setting in for travelers worl...


In [13]:
def get_vader_sentiment(text):
    if pd.isna(text) or text == "None":
        return None
    return analyzer.polarity_scores(str(text))['compound']

news_df_cleaned['content_sentiment'] = news_df_cleaned['content'].apply(get_vader_sentiment)

print(news_df_cleaned[['title','content_sentiment']])

                                                title  content_sentiment
0   Is Anthropic limiting the release of Mythos to...             0.1779
1   'Big Short' investor Michael Burry says Anthro...             0.9022
2   Exclusive | Sazerac Eyes Deal With Jack Daniel...                NaN
3   SpaceX Woos Investors As C-Suite Shakeups Cont...                NaN
4   Amazon Rises Toward Key Level; CEO Says This I...             0.2960
5   David Zaslav Paramount Deal 'Windfall' Payout:...             0.4404
6   US Postal Service to suspend employer payments...            -0.3182
7   Mark Zuckerberg announces Muse Spark, a new Me...                NaN
8   'No idea what he’s thinking': New policy at Ph...             0.0000
9   Travelers face higher costs and fewer flight o...            -0.1531
10  Mortgage rates fall on Iran ceasefire: Mortgag...             0.4767
11  Stocks gain for a second day on hope Iran ceas...             0.7269
12  Kia plans to launch U.S. pickup truck by 2030 .

In [14]:
news_df_cleaned['content_sentiment'].mean()

np.float64(0.14351428571428573)

Тут у нас он получился меньше чем в прошлый раз, тут мы собирали просто новости по всему бизнесу в целом.


In [15]:
#3-й запрос
import requests
url = ('https://newsapi.org/v2/top-headlines?''country=us&''category=business&''apiKey=1fd00ba235bf47278e2c13643565a3ce')
response = requests.get(url)
res = response.json()
news_df = pd.DataFrame(res['articles'])
news_df = news_df.drop('urlToImage', axis=1)



In [16]:

def clean_page_columns(news_df):
    news_df['source'] = news_df['source'].apply(lambda d: d['name'])
    return news_df[['source', 'author', 'title', 'description', 'url', 'publishedAt', 'content']]

In [17]:
page = pd.DataFrame(res['articles'])
news_df_cleaned = clean_page_columns(page)
news_df_cleaned

,source,author,title,description,url,publishedAt,content
0,TechCrunch,Tim Fernholz,Is Anthropic limiting the release of Mythos to...,Anthropic said this week that it limited the r...,https://techcrunch.com/2026/04/09/is-anthropic...,2026-04-09T18:50:05Z,Anthropic said this week that it limited the r...
1,Business Insider,Samuel O'Brient,'Big Short' investor Michael Burry says Anthro...,Palantir's stock is down. Michael Burry said W...,https://www.businessinsider.com/michael-burry-...,2026-04-09T18:43:00Z,Michael Burry has long been bearish on Palanti...
2,The Wall Street Journal,Laura Cooper,Exclusive | Sazerac Eyes Deal With Jack Daniel...,Interest comes as Pernod Ricard has been discu...,https://www.wsj.com/business/sazerac-eyes-deal...,2026-04-09T18:05:00Z,None
3,The Information,"Theo Wayt, Cory Weinberg, Valida Pau",SpaceX Woos Investors As C-Suite Shakeups Cont...,Some of the biggest Wall Street fund managers ...,https://www.theinformation.com/articles/spacex...,2026-04-09T17:53:00Z,None
4,Investor's Business Daily,None,Amazon Rises Toward Key Level; CEO Says This I...,Andy Jassy touted growth for the tech giant's ...,https://www.investors.com/news/technology/amaz...,2026-04-09T17:52:00Z,Information in Investors Business Daily is for...
5,Variety,Todd Spangler,David Zaslav Paramount Deal 'Windfall' Payout:...,ISS recommends Warner Bros. Discovery sharehol...,https://variety.com/2026/film/news/david-zasla...,2026-04-09T17:47:00Z,Does David Zaslav deserve to receive more than...
6,Associated Press,Susan Haigh,US Postal Service to suspend employer payments...,The U.S. Postal Service has decided to tempora...,https://apnews.com/article/mail-usps-pensions-...,2026-04-09T17:13:00Z,The U.S. Postal Service said Thursday it has i...
7,Mashable,"Chance Townsend, Timothy Beck Werth","Mark Zuckerberg announces Muse Spark, a new Me...",Nine months after founding Meta Superintellige...,https://mashable.com/article/mark-zuckerberg-m...,2026-04-09T17:06:34Z,None
8,SFGate,Ariana Bindman,'No idea what he’s thinking': New policy at Ph...,"Philz, the coffee chain that was founded in Sa...",https://www.sfgate.com/food/article/philz-outr...,2026-04-09T16:25:29Z,Customers enter Philz Coffee at Bay Street Eme...
9,Associated Press,"Rio Yamat, Ap Airlines, Travel Writer",Travelers face higher costs and fewer flight o...,Air travelers are facing a new reality of high...,https://apnews.com/article/airline-tickets-fee...,2026-04-09T16:22:00Z,A new reality is setting in for travelers worl...


In [18]:
def get_vader_sentiment(text):
    if pd.isna(text) or text == "None":
        return None
    return analyzer.polarity_scores(str(text))['compound']

news_df_cleaned['content_sentiment'] = news_df_cleaned['content'].apply(get_vader_sentiment)

print(news_df_cleaned[['title','content_sentiment']])

                                                title  content_sentiment
0   Is Anthropic limiting the release of Mythos to...             0.1779
1   'Big Short' investor Michael Burry says Anthro...             0.9022
2   Exclusive | Sazerac Eyes Deal With Jack Daniel...                NaN
3   SpaceX Woos Investors As C-Suite Shakeups Cont...                NaN
4   Amazon Rises Toward Key Level; CEO Says This I...             0.2960
5   David Zaslav Paramount Deal 'Windfall' Payout:...             0.4404
6   US Postal Service to suspend employer payments...            -0.3182
7   Mark Zuckerberg announces Muse Spark, a new Me...                NaN
8   'No idea what he’s thinking': New policy at Ph...             0.0000
9   Travelers face higher costs and fewer flight o...            -0.1531
10  Mortgage rates fall on Iran ceasefire: Mortgag...             0.4767
11  Stocks gain for a second day on hope Iran ceas...             0.7269
12  Kia plans to launch U.S. pickup truck by 2030 .

In [19]:
#4-й запрос
import requests
url = ('https://newsapi.org/v2/everything?''q=bitcoin&''sortBy=popularity&''apiKey=1fd00ba235bf47278e2c13643565a3ce')
response = requests.get(url)
res = response.json()
news_df = pd.DataFrame(res['articles'])
news_df = news_df.drop('urlToImage', axis=1)
def clean_page_columns(news_df):
    news_df['source'] = news_df['source'].apply(lambda d: d['name'])
    return news_df[['source', 'author', 'title', 'description', 'url', 'publishedAt', 'content']]
page = pd.DataFrame(res['articles'])
news_df_cleaned = clean_page_columns(page)
news_df_cleaned

,source,author,title,description,url,publishedAt,content
0,Wired,Kate Knibbs,Wall Street Is Already Betting on Prediction M...,As the legal war over how to regulate predicti...,https://www.wired.com/story/prediction-markets...,2026-03-16T09:30:00Z,When Troy Dixon first suggested incorporating ...
1,Wired,Matthew S. Smith,Identity Theft Protection Services: Do You Act...,Here's the best advice I have for protecting y...,https://www.wired.com/story/best-id-protection...,2026-03-14T10:30:00Z,Odds are good youve encountered an identity th...
2,Gizmodo.com,Kyle Torpey,UK Man Accuses Spouse of Stealing $172 Million...,"Code is law, some say. That doesn't mean norma...",https://gizmodo.com/uk-man-accuses-spouse-of-s...,2026-03-17T22:47:01Z,"Ping Fai Yuen, who is a U.K. resident, has acc..."
3,Gizmodo.com,Bruce Gil,The New York Times Claims It Finally Unmasked ...,"If true, the man would be one of the richest p...",https://gizmodo.com/the-new-york-times-claims-...,2026-04-08T16:40:08Z,"The real identity of Satoshi Nakamoto, the cre..."
4,Gizmodo.com,Kyle Torpey,"Gemini, Crypto.com Latest Crypto Firms to Blam...","Meanwhile, some companies have abandoned crypt...",https://gizmodo.com/gemini-crypto-com-latest-c...,2026-03-21T17:00:54Z,With the bitcoin price still sitting roughly 4...
...,...,...,...,...,...,...,...
95,Habr.com,Azamat_Safarov,Блокчейн как инфраструктура E-Health: новая мо...,Представьте: вы обращаетесь в три разные клини...,https://habr.com/ru/articles/1014024/#post-con...,2026-03-24T06:00:40Z,": . , . . . .\r\n . : , , , , .\r\n a. IBM Cos..."
96,Habr.com,arhip1986,"Не биты, а тетраэдры: как я построил геометрич...","Мы привыкли думать о вычислениях как о битах, ...",https://habr.com/ru/articles/1011646/#post-con...,2026-03-18T07:54:16Z,", . , , ? , , , jump- RTX 3090 exact- 554.92 ...."
97,Techmeme.com,None,Fannie Mae will accept crypto-backed mortgages...,Wall Street Journal:\nFannie Mae will accept c...,https://www.techmeme.com/260326/p35,2026-03-26T18:40:00Z,About This Page\r\nThis is a Techmeme archive ...
98,Techmeme.com,None,A look at Coinbase One and other insurance-lik...,Bloomberg:\nA look at Coinbase One and other i...,https://www.techmeme.com/260329/p8,2026-03-29T19:40:14Z,About This Page\r\nThis is a Techmeme archive ...


In [20]:
news_df_cleaned['content_sentiment'] = news_df_cleaned['content'].apply(get_vader_sentiment)

print(news_df_cleaned[['title','content_sentiment']])

                                                title  content_sentiment
0   Wall Street Is Already Betting on Prediction M...            -0.3400
1   Identity Theft Protection Services: Do You Act...             0.4404
2   UK Man Accuses Spouse of Stealing $172 Million...            -0.6124
3   The New York Times Claims It Finally Unmasked ...             0.1280
4   Gemini, Crypto.com Latest Crypto Firms to Blam...            -0.2263
..                                                ...                ...
95  Блокчейн как инфраструктура E-Health: новая мо...             0.3400
96  Не биты, а тетраэдры: как я построил геометрич...             0.0000
97  Fannie Mae will accept crypto-backed mortgages...             0.0000
98  A look at Coinbase One and other insurance-lik...             0.0000
99  Acheter du Bitcoin quand son prix baisse : opp...             0.0000

[100 rows x 2 columns]


In [21]:
news_df_cleaned['content_sentiment'].mean()

np.float64(0.027602999999999996)

В новостях про биткоин выявить тенденции не удалось.


In [22]:
#5-й запрос
import requests
url = ('https://newsapi.org/v2/everything?''q=finance&''sortBy=popularity&''apiKey=1fd00ba235bf47278e2c13643565a3ce')
response = requests.get(url)
res = response.json()
news_df = pd.DataFrame(res['articles'])
news_df = news_df.drop('urlToImage', axis=1)
def clean_page_columns(news_df):
    news_df['source'] = news_df['source'].apply(lambda d: d['name'])
    return news_df[['source', 'author', 'title', 'description', 'url', 'publishedAt', 'content']]
page = pd.DataFrame(res['articles'])
news_df_cleaned = clean_page_columns(page)
news_df_cleaned

,source,author,title,description,url,publishedAt,content
0,Wired,Kate Knibbs,Wall Street Is Already Betting on Prediction M...,As the legal war over how to regulate predicti...,https://www.wired.com/story/prediction-markets...,2026-03-16T09:30:00Z,When Troy Dixon first suggested incorporating ...
1,The Verge,Terrence O’Brien,AI Czar David Sacks wants Trump to ‘get out’ o...,"David Sacks, the White House's AI and crypto c...",https://www.theverge.com/policy/895059/trump-a...,2026-03-15T14:13:25Z,<ul><li></li><li></li><li></li></ul>\r\nIt mig...
2,The Verge,Elizabeth Lopatto,"Oh, you think the government will regulate Kal...",The Commodity Futures Trading Commission has a...,https://www.theverge.com/business/896517/kalsh...,2026-03-18T14:20:24Z,"<ul><li></li><li></li><li></li></ul>\r\nOh, yo..."
3,The Verge,Josh Dzieza,You Could Be Next,The LinkedIn post seemed like yet another scam...,https://www.theverge.com/cs/features/877388/wh...,2026-03-10T09:00:01Z,The LinkedIn post seemed like yet another scam...
4,BBC News,Ben Ramsdale,Survival or FA Cup glory - which would you cho...,Leeds visit West Ham on Sunday in the FA Cup q...,https://www.bbc.com/sport/football/articles/c1...,2026-04-05T05:25:09Z,The winner of this year's men's FA Cup will po...
...,...,...,...,...,...,...,...
94,Xataka.com,Rubén Andrés,Irán se fijó como objetivo acabar con Dubái: l...,"Durante años, Dubái ha sido la tierra prometid...",https://www.xataka.com/empresas-y-economia/dub...,2026-03-23T19:16:54Z,"Durante años, Dubái ha sido la tierra prometid..."
95,Xataka.com,Javier Marquez,Yuanjie es la desconocida empresa china de tec...,Yuanjie Semiconductor Technology probablemente...,https://www.xataka.com/empresas-y-economia/yua...,2026-03-23T22:00:54Z,Yuanjie Semiconductor Technology probablemente...
96,Themarginalian.org,Maria Popova,Create Dangerously: Albert Camus on the Power ...,"""To create today is to create dangerously... T...",https://www.themarginalian.org/2026/03/24/albe...,2026-03-24T14:29:33Z,“Those who tell you Do not put too much politi...
97,Abcnews.com,Max Zahn,"Stocks close higher, reversing sharp losses af...",Oil prices spiked as high as nearly $120 per b...,https://abcnews.com/Business/stocks-tumble-oil...,2026-03-09T20:21:36Z,"Stocks closed higher on Monday, recovering fro..."


In [23]:
news_df_cleaned['content_sentiment'] = news_df_cleaned['content'].apply(get_vader_sentiment)

print(news_df_cleaned[['title','content_sentiment']])

                                                title  content_sentiment
0   Wall Street Is Already Betting on Prediction M...            -0.3400
1   AI Czar David Sacks wants Trump to ‘get out’ o...            -0.9325
2   Oh, you think the government will regulate Kal...             0.1306
3                                   You Could Be Next            -0.5927
4   Survival or FA Cup glory - which would you cho...             0.8860
..                                                ...                ...
94  Irán se fijó como objetivo acabar con Dubái: l...            -0.2960
95  Yuanjie es la desconocida empresa china de tec...            -0.2960
96  Create Dangerously: Albert Camus on the Power ...             0.6115
97  Stocks close higher, reversing sharp losses af...            -0.7650
98  Treasury taking over federal student loans ami...             0.2023

[99 rows x 2 columns]


In [ ]:
news_df_cleaned['content_sentiment'].mean()

np.float64(0.02403434343434344)

Тут мы опять получили нулевой сентимент.


In [24]:
#6-й запрос
import requests
url = ('https://newsapi.org/v2/everything?''q=акции&''language=ru&''sortBy=popularity&''apiKey=1fd00ba235bf47278e2c13643565a3ce')
response = requests.get(url)
res = response.json()
news_df = pd.DataFrame(res['articles'])
news_df = news_df.drop('urlToImage', axis=1)
def clean_page_columns(news_df):
    news_df['source'] = news_df['source'].apply(lambda d: d['name'])
    return news_df[['source', 'author', 'title', 'description', 'url', 'publishedAt', 'content']]
page = pd.DataFrame(res['articles'])
news_df_cleaned = clean_page_columns(page)
news_df_cleaned

,source,author,title,description,url,publishedAt,content
0,Habr.com,klimensky (FirstVDS),"Как Apple едва не уничтожила себя, разрешив кл...",В 1997 году акции Apple стоили 4 доллара. Комп...,https://habr.com/ru/companies/first/articles/1...,2026-03-20T09:40:54Z,"Apple : . , , .\r\n 1994 Apple . Mac «» .\r\n ..."
1,Lifehacker.ru,Мила Цимбал,«Великая китайская распродажа» на AliExpress: ...,"Изучили ассортимент и собрали товары, на котор...",https://lifehacker.ru/vkr-mart-2026/,2026-03-13T11:00:00Z,": 16 «» AliExpress. , , , , 90%. , : .\r\n , -..."
2,Lenta,Кирилл Луцюк,Война с Ираном обрушила акции европейских комп...,Ближневосточный кризис нанес еще один удар по ...,https://lenta.ru/news/2026/04/07/voyna-s-irano...,2026-04-07T12:36:19Z,". Bloomberg.\r\n, Euro Stoxx 50, 50 12 , . S&a..."
3,Lenta,Лилиана Набиуллина,Акции Сбера ускорили рост после слов Грефа о п...,Акции Сбера отреагировали ростом на заявление ...,https://lenta.ru/news/2026/03/17/aktsii-sbera-...,2026-03-17T13:00:52Z,"«» , . 15:10 0,82 , 318,98 , Investing.com.\r\..."
4,Lenta,Алена Шевченко,Россиянам развеяли два популярных мифа об орга...,Качественное органическое вино не имеет дрожже...,https://lenta.ru/news/2026/04/05/rossiyanam-ra...,2026-04-05T04:30:54Z,", , . «.» « ».\r\n , . , , , , , .\r\n« , , ...."
...,...,...,...,...,...,...,...
95,BBC News,https://www.facebook.com/bbcnews,В США проходят акции протеста против Дональда ...,В городах по всей территории США проходят масш...,https://www.bbc.com/russian/articles/cde5lpde4pko,2026-03-29T07:16:19Z,", : « »\r\n« ». , . \r\n , , , , .\r\n« . , , ..."
96,BBC News,None,Как война в Иране бьет по рейтингам Трампа?,"В этом выпуске поговорим о том, как война в Ир...",https://www.bbc.com/russian/podcasts/p076qqzl/...,2026-03-31T00:00:00Z,", . . « !». , .\r\n .\r\n00:00 . \r\n00:44 ?\r..."
97,BBC News,None,Митинги в защиту «Телеграма»: протест или пров...,На 29 марта в России анонсированы протесты про...,https://www.bbc.com/russian/podcasts/p076qqzl/...,2026-03-26T14:33:00Z,"29 . , . , . 1 «». ? ? «» ? -- .\r\n00:00 . \r..."
98,Interfax.ru,None,Суд присудил Газпрому акции ликвидированных юр...,"Интерфакс: ""Газпром"" в суде оформил собственно...",https://www.interfax.ru/business/1078081,2026-03-16T04:12:00Z,". 16 . INTERFAX.RU - """" 342 . - . , 2024-2025 ..."


In [25]:
news_df_cleaned['content_sentiment'] = news_df_cleaned['content'].apply(get_vader_sentiment)

print(news_df_cleaned[['title','content_sentiment']])

                                                title  content_sentiment
0   Как Apple едва не уничтожила себя, разрешив кл...                0.0
1   «Великая китайская распродажа» на AliExpress: ...                0.0
2   Война с Ираном обрушила акции европейских комп...                0.0
3   Акции Сбера ускорили рост после слов Грефа о п...                0.0
4   Россиянам развеяли два популярных мифа об орга...                0.0
..                                                ...                ...
95  В США проходят акции протеста против Дональда ...                0.0
96        Как война в Иране бьет по рейтингам Трампа?                0.0
97  Митинги в защиту «Телеграма»: протест или пров...                0.0
98  Суд присудил Газпрому акции ликвидированных юр...                0.0
99  Производитель игрушек Labubu в 2025 году нарас...                0.0

[100 rows x 2 columns]


А на русскоязычных данных он не работает.


Что мы получили по итогу, сентимент мы оценивать можем только для english, следующая итерация. Мы хотим проверить как влияет и влияет ли сентимент на изменение цены акции, для более эффективной торговли.